# Counterbalancing

Same steps as the original `counterbalance.ipynb`, but the shared machinery comes from
`stimuli_pipeline` instead of `%run design_stimuli.ipynb`.

`build_context()` rebuilds exactly what the `%run` used to leave in the namespace
(`nodes_en`, `unrel_en`, `unrel_zh2en`, `shared_translation`, `wpos`, the RW/strength
matrices, ...) — now on a single `ctx` object. RW matrices are cached to disk, so this
is fast on repeat runs. Nothing from the selection stage or the permutation test is
re-executed, because counterbalancing does not need it.

Everything below can also be run in one line from a terminal:

```bash
python counterbalance.py --help
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))   # the folder holding stimuli_pipeline/

from stimuli_pipeline import Config, build_context
from stimuli_pipeline.counterbalance import (
    load_reviewed, patch_reviewed_metrics, uniqueness_check, rematch_minus_minus,
    build_lists, qc_lists, save_full_design,
)
from stimuli_pipeline.utils import log

# Any parameter can be overridden here, e.g. Config(dist_cutoff=1.4, seed=123)
cfg = Config()
log(f"outputs will go to {cfg.out_dir}")

In [ ]:
ctx = build_context(cfg)

# Global Uniqueness Check

In [ ]:
v_design = load_reviewed(cfg)     # manual_review/uniqueness_checked3.csv
print(v_design['cue_en'].nunique(), 'cues')
v_design.info()

In [ ]:
# Manual review can leave blank cells: rows pasted from matched_pairs_reordered.csv
# carry no str_zh, because stage 1 writes that quantity as route_s_zh. Refill them
# from the matrices -- str_zh comes from the route recorded on the row, falling back
# to the recovered winning route. Existing values are never overwritten, and M-E+/M-E-
# rows stay blank because they have no Mandarin route.
v_design = patch_reviewed_metrics(ctx, v_design)
v_design.isna().sum().loc[lambda s: s > 0]

In [ ]:
# Flags words used as a target in more than one pair, or as both cue and target.
# Counts each cue once (not once per condition) and each cue-target pair once.
# Runs BEFORE the M-E- rematch, so it reflects the 3-cell set only -- the rematch
# deliberately reuses M+E+ targets afterwards, which is by design.
v_design = uniqueness_check(v_design, cfg)   # writes output/uniqueness_issue4.csv

# Rematch for M-E-

Reuse the M+E+ target pool as the M-E- pool, so the critical and baseline cells share
the same vocabulary and word frequency/length/concreteness cannot confound the
comparison. `cfg.max_total_target_uses` caps how many (cue, condition) slots any one
word may occupy across the whole design, leaving the Latin square a list of margin.

In [ ]:
full_design = rematch_minus_minus(ctx, v_design)
full_design.tail()

# Counterbalanced Lists

Build 4 Latin-square lists (A-D). Every participant sees all 60 cues exactly once;
which of the 4 conditions a cue is tested in rotates across lists, so across the 4
lists each cue appears once in every condition. Within a single list no target word
may repeat (global uniqueness *per list*, not across the whole design -- the M+E+/M-E-
pool sharing means the same word legitimately serves two different cues' target lists
overall, but never within the same list for two different cues).

Cues are split into 4 groups of 15. In list `L`, a cue from group `g` is shown at
condition `CONDITIONS[(g + L) % 4]`. The group assignment is found by simulated
annealing, minimizing (a) within-list target collisions and (b) group-size imbalance,
until zero collisions are found.

In [ ]:
lists_df, groups = build_lists(full_design, cfg)
lists_df.head()

In [ ]:
# ---- QC: verify the counterbalanced lists ----
checks_passed = qc_lists(lists_df, cfg)

# Save Final Output

`full_design.xlsx` has 5 sheets:
- `all_pairs`: all 240 cue-target pairs (the full metric set from `full_design`, minus
  the manual `Issue`/duplicate-flag columns), plus a `list` column marking which of the
  4 counterbalanced lists shows that (cue, condition) pairing.
- `List A`-`List D`: the 60 rows shown to participants assigned to that list, with
  the full metric columns and `stim_id` carried over from `all_pairs`.

In [ ]:
all_pairs, out_path = save_full_design(full_design, lists_df, cfg)
all_pairs.head()

---
# Appendix

## The whole stage in one call

Equivalent to every cell above (`run_counterbalance` also loads the reviewed file if
you don't pass one).

In [ ]:
# from stimuli_pipeline.counterbalance import run_counterbalance
# result = run_counterbalance(ctx)
# result['lists_df'].head()

## Re-running the selection stage

The design stage (`design_stimuli.py`) is available from the same context, in one call
or step by step, if you want to inspect the candidate pairs here.

In [ ]:
# from stimuli_pipeline.selection import (rank_targets, add_similarity, load_norms,
#                                         add_covariate_distance, match_pairs,
#                                         largest_cue_clique, build_design)
#
# all_cand = rank_targets(ctx)
# all_cand, similarity = add_similarity(ctx, all_cand)
# fit = add_covariate_distance(all_cand, load_norms(cfg))
# matched = match_pairs(fit, cfg)
# design = build_design(matched, largest_cue_clique(matched, fit, cfg), cfg)
#
# sns.histplot(data=all_cand[all_cand['condition'] != 'M-E+'],
#              x='str_zh, alignment weighted', hue='condition')

In [ ]:
# sns.histplot(fit.filtered_pairs, x='cov_distance')